Normalizing the chapters with the decided Agents

In [ ]:
"""
Normalization of La Bête humaine, starting from the RAW chapter files.

Two kinds of rules (edit them in the CONFIGURATION section below):

  - TO_CHECK: ambiguous expressions ("la machine", "le mécanicien"...).
    Every occurrence goes into a CSV file that you annotate by hand
    (replace = yes / no). Only the "yes" are replaced.
  - ALWAYS: safe aliases ("madame Roubaud" -> Séverine...), always replaced.

Commands (command line: python normalize_lbh.py extract ...;
in a notebook: main(["extract", ...])):

1) extract: list every TO_CHECK occurrence, with context, in a CSV.
       extract --raw raw_chapters --csv to_annotate.csv
   Optional: --previous machine-to-annotate.csv re-uses your earlier
   annotations of "la machine" / "sa machine" (column is_lison).

2) summary: count yes / no / empty per rule and chapter.
       summary --csv to_annotate.csv

3) apply: write the normalized chapters.
       apply --raw raw_chapters --csv to_annotate.csv --out normalized_chapters

Do not edit the raw files between "extract" and "apply",
otherwise the IDs no longer match.
"""

import argparse
import csv
import re
import sys
from collections import Counter
from pathlib import Path

# ============================ CONFIGURATION ============================

# Ambiguous expressions, checked by hand.
# key: (regex, replacement function, target name)
# The regex is case-insensitive; group 1 is the article / determiner.


def lison(m):
    # keep the determiner: "la machine" -> "la Lison", "sa machine" -> "sa Lison"
    return f"{m.group(1)} Lison"


def jacques(m):
    art = m.group(1).lower()
    # keep the sentence grammatical: "du mécanicien" -> "de Jacques"
    rep = {"du": "de Jacques", "au": "à Jacques"}.get(art, "Jacques")
    return rep[0].upper() + rep[1:] if m.group(1)[0].isupper() else rep


TO_CHECK = {
    "machine": (r"\b(la|sa)\s+(machine)\b", lison, "Lison"),
    "mecanicien": (r"\b(le|du|au|son)\s+(mécanicien)\b", jacques, "Jacques"),
}

# Safe aliases, always replaced, in this order (longest forms first).
# Adapt this list to the rules of your previous normalization script.
ALWAYS = [
    (r"\b(?:madame|Madame|Mme|mme)\s+Roubaud\b", "Séverine"),
    (r"\bJacques\s+Lantier\b", "Jacques"),
    (r"\btante\s+Phasie\b", "Phasie"),
]

# ======================================================================

CONTEXT = 90
FIELDS = ["id", "rule", "target", "chapter", "replace", "left", "form", "right", "note"]
YES = {"yes", "y", "oui", "o", "1"}
NO = {"no", "n", "non", "0"}
COMPILED = {k: (re.compile(p, re.IGNORECASE), f, t) for k, (p, f, t) in TO_CHECK.items()}


def chapter_files(folder):
    files = list(Path(folder).glob("*.txt"))
    if not files:
        sys.exit(f"No .txt files in {folder}")

    def number(f):
        digits = re.findall(r"\d+", f.stem)
        return int(digits[-1]) if digits else 0

    return sorted(files, key=number), number


def flatten(s):
    return re.sub(r"\s+", " ", s).strip()


def make_id(rule, chapter, i):
    return f"{rule}_ch{chapter:02d}_{i:03d}"


def label(value):
    v = (value or "").strip().lower()
    return "yes" if v in YES else "no" if v in NO else "empty"


def read_csv(path):
    with open(path, encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f, delimiter=";"))


def extract(args):
    previous = {}
    if args.previous:
        # earlier file from verif_machine.py: ids "ch01_001", column is_lison
        for r in read_csv(args.previous):
            previous["machine_" + r["id"]] = (r.get("is_lison", ""), r.get("note", ""))
    files, number = chapter_files(args.raw)
    rows = []
    for f in files:
        text = f.read_text(encoding="utf-8")
        chapter = number(f)
        for rule, (rx, _, target) in COMPILED.items():
            for i, m in enumerate(rx.finditer(text), 1):
                rid = make_id(rule, chapter, i)
                rep, note = previous.get(rid, ("", ""))
                rows.append({
                    "id": rid, "rule": rule, "target": target, "chapter": chapter,
                    "replace": rep, "note": note,
                    "left": flatten(text[max(0, m.start() - CONTEXT):m.start()]),
                    "form": flatten(m.group(0)),
                    "right": flatten(text[m.end():m.end() + CONTEXT]),
                })
    with open(args.csv, "w", encoding="utf-8-sig", newline="") as out:
        w = csv.DictWriter(out, fieldnames=FIELDS, delimiter=";")
        w.writeheader()
        w.writerows(rows)
    counts = Counter(r["rule"] for r in rows)
    reused = sum(1 for r in rows if r["replace"])
    print(f"{len(rows)} occurrences written to {args.csv}")
    for rule in TO_CHECK:
        print(f"  {rule}: {counts[rule]}")
    if args.previous:
        print(f"  annotations re-used from {args.previous}: {reused}")


def summary(args):
    stats = {}
    for r in read_csv(args.csv):
        key = (r["rule"], int(r["chapter"]), r["form"].lower())
        stats.setdefault(key, Counter())[label(r["replace"])] += 1
    totals = {}
    print("rule ; chapter ; form ; yes ; no ; not annotated")
    for key in sorted(stats):
        s = stats[key]
        totals.setdefault(key[0], Counter()).update(s)
        print(f"{key[0]} ; {key[1]:2d} ; {key[2]} ; {s['yes']} ; {s['no']} ; {s['empty']}")
    print()
    for rule, s in totals.items():
        print(f"TOTAL {rule} ; yes {s['yes']} ; no {s['no']} ; not annotated {s['empty']}")
    empty = sum(s["empty"] for s in totals.values())
    if empty:
        print(f"\nWarning: {empty} occurrences not annotated yet (they will NOT be replaced).")


def apply(args):
    rows = read_csv(args.csv)
    validated = {r["id"] for r in rows if label(r["replace"]) == "yes"}
    expected_left = {r["id"]: r.get("left", "") for r in rows}
    mismatches = []
    files, number = chapter_files(args.raw)
    out_dir = Path(args.out)
    out_dir.mkdir(parents=True, exist_ok=True)
    checked = Counter()
    always = Counter()
    for f in files:
        text = f.read_text(encoding="utf-8")
        chapter = number(f)
        # 1) find all validated spans on the RAW text, then replace in one pass
        spans = []
        for rule, (rx, func, _) in COMPILED.items():
            for i, m in enumerate(rx.finditer(text), 1):
                rid = make_id(rule, chapter, i)
                if rid in validated:
                    # safety check: the context in the CSV must match the raw text
                    left = flatten(text[max(0, m.start() - CONTEXT):m.start()])
                    exp = expected_left.get(rid, "")
                    if exp and exp != left:
                        mismatches.append(rid)
                        continue
                    spans.append((m.start(), m.end(), func(m), rule))
        spans.sort()
        parts, pos = [], 0
        for start, end, rep, rule in spans:
            if start < pos:  # overlapping matches: keep the first one
                continue
            parts += [text[pos:start], rep]
            pos = end
            checked[rule] += 1
        text = "".join(parts) + text[pos:]
        # 2) safe aliases
        for pattern, rep in ALWAYS:
            text, n = re.subn(pattern, rep, text)
            always[pattern] += n
        (out_dir / f.name).write_text(text, encoding="utf-8")
    print("Checked rules (replaced):")
    for rule in TO_CHECK:
        print(f"  {rule}: {checked[rule]}")
    print("Safe aliases (replaced):")
    for pattern, rep in ALWAYS:
        print(f"  {pattern} -> {rep}: {always[pattern]}")
    print(f"Normalized chapters written to {out_dir}/")
    if mismatches:
        print(f"\nWARNING: {len(mismatches)} rows skipped, their context does not match "
              f"the raw text (check them): {', '.join(mismatches)}")
    else:
        print("Context check: all validated rows match the raw text.")


def main(argv=None):
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = p.add_subparsers(dest="command", required=True)
    e = sub.add_parser("extract")
    e.add_argument("--raw", required=True)
    e.add_argument("--csv", required=True)
    e.add_argument("--previous", help="earlier machine annotations (verif_machine.py)")
    s = sub.add_parser("summary")
    s.add_argument("--csv", required=True)
    a = sub.add_parser("apply")
    a.add_argument("--raw", required=True)
    a.add_argument("--csv", required=True)
    a.add_argument("--out", required=True)
    args = p.parse_args(argv)
    {"extract": extract, "summary": summary, "apply": apply}[args.command](args)


if __name__ == "__main__" and "ipykernel" not in sys.modules:
    main()

In [ ]:
main(["apply", "--raw", "chapter_files", "--csv", "annotation-before-normalisation.csv", "--out", "corrected_chapters"])

Checked rules (replaced):
  machine: 56
  mecanicien: 48
Safe aliases (replaced):
  \b(?:madame|Madame|Mme|mme)\s+Roubaud\b -> Séverine: 13
  \bJacques\s+Lantier\b -> Jacques: 7
  \btante\s+Phasie\b -> Phasie: 17
Normalized chapters written to corrected_chapters/
Context check: all validated rows match the raw text.


In [ ]:
import shutil
from google.colab import files

# zip the normalized chapters
shutil.make_archive("corrected_chapters", "zip", "corrected_chapters")

# download the zip and the annotation file
files.download("corrected_chapters.zip")
files.download("to_annotate.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Keywords - Creating nodes for Gephi

In [ ]:
"""
Keyword occurrences of La Bête humaine -> annotation table -> Gephi files.

Use the RAW chapter files (same folder as for normalize_lbh.py).

Commands (command line: python keywords_table.py extract ...;
in a notebook: main(["extract", ...])):

1) extract: every occurrence of the selected keywords, with context, in a CSV.
       extract --raw chapter_files --csv keywords_to_annotate.csv

   Columns to fill by hand:
     applies_to : who / what the word describes or concerns
                  (Lison, Séverine, Flore, Philomène, Phasie, Louisette,
                   other_machine, other)
     agent      : who does the action, if any (e.g. Jacques for "posséder")
     role       : optional, transitivity role of applies_to
                  (Actor, Goal, Senser, Carrier, Phenomenon...)
     note       : free text

2) summary: counts per keyword x applies_to, and per phase.
       summary --csv keywords_to_annotate.csv

3) gephi: write nodes and edges for Gephi from the annotated CSV.
       gephi --csv keywords_to_annotate.csv --out gephi_files
   - nodes_words.csv      Id, Label, type (word / character), for edges_words.csv
   - nodes_agents.csv     characters only, for edges_agents.csv
   - edges_words.csv      word -- applies_to (weight = occurrences),
                          with one column per phase
   - edges_agents.csv     agent -> applies_to, label = keyword (directed)
   Import in Gephi: Data Laboratory > Import Spreadsheet, nodes first.
   Option: --exclude other,men_generic,women_generic (default: other).
"""

import argparse
import csv
import re
import sys
from collections import Counter, defaultdict
from pathlib import Path

# ============================ CONFIGURATION ============================

# Selected keywords (from the AntConc keyword list against the 15 novels).
# key: regex of the forms counted (case-insensitive, whole words).
KEYWORDS = {
    "docile": r"dociles?",
    "cavale": r"cavales?",
    "rétif": r"rétive?s?|rétifs?",
    "mâle": r"mâles?",
    # verb and noun merged into one group (same family)
    "posséder/possession": r"posséd\w*|possèd\w*|possessions?",
    "désir": r"désirs?",
    "chéri": r"chérie?s?",
    "jaloux": r"jaloux|jalouses?",
    # tuer, tué, tuerait... merged into one group
    "tuer": r"tu(?:er|é|ée|és|ées|e|es|ent|ait|aient|ais|a|ât|èrent|erai|erais|erait|eraient|eras?|era|erons|erez|eront|ant|ons|ez)",
    "morte": r"mortes?",
    "sang": r"sangs?",
}

# Semantic group of each keyword (node attribute in Gephi, for colouring).
WORD_GROUP = {
    "docile": "animal / temperament", "cavale": "animal / temperament",
    "rétif": "animal / temperament", "mâle": "animal / temperament",
    "posséder/possession": "desire / possession", "désir": "desire / possession",
    "chéri": "desire / possession", "jaloux": "desire / possession",
    "tuer": "violence / death", "morte": "violence / death", "sang": "violence / death",
}

# Phases: Lison's three main chapters (concordance plot).
PHASES = {5: "P1_ch5_run", 7: "P2_ch7_snow", 10: "P3_ch10_death"}
OTHER_PHASE = "other_chapters"

# ======================================================================

CONTEXT = 120
FIELDS = ["id", "keyword", "chapter", "phase", "left", "form", "right",
          "applies_to", "agent", "role", "note"]
COMPILED = {k: re.compile(rf"\b(?:{p})\b", re.IGNORECASE) for k, p in KEYWORDS.items()}


def chapter_files(folder):
    files = list(Path(folder).glob("*.txt"))
    if not files:
        sys.exit(f"No .txt files in {folder}")

    def number(f):
        digits = re.findall(r"\d+", f.stem)
        return int(digits[0]) if digits else 0

    return sorted(files, key=number), number


def flatten(s):
    return re.sub(r"\s+", " ", s).strip()


def read_csv(path):
    with open(path, encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f, delimiter=";"))


def clean(v):
    return (v or "").strip()


def extract(args):
    files, number = chapter_files(args.raw)
    rows = []
    for f in files:
        text = f.read_text(encoding="utf-8")
        ch = number(f)
        for kw, rx in COMPILED.items():
            for i, m in enumerate(rx.finditer(text), 1):
                rows.append({
                    "id": f"{kw}_ch{ch:02d}_{i:03d}", "keyword": kw, "chapter": ch,
                    "phase": PHASES.get(ch, OTHER_PHASE),
                    "left": flatten(text[max(0, m.start() - CONTEXT):m.start()]),
                    "form": m.group(0),
                    "right": flatten(text[m.end():m.end() + CONTEXT]),
                    "applies_to": "", "agent": "", "role": "", "note": "",
                })
    rows.sort(key=lambda r: (list(KEYWORDS).index(r["keyword"]), r["chapter"], r["id"]))
    with open(args.csv, "w", encoding="utf-8-sig", newline="") as out:
        w = csv.DictWriter(out, fieldnames=FIELDS, delimiter=";")
        w.writeheader()
        w.writerows(rows)
    print(f"{len(rows)} occurrences written to {args.csv}")
    c = Counter(r["keyword"] for r in rows)
    for kw in KEYWORDS:
        print(f"  {kw}: {c[kw]}")


def summary(args):
    rows = read_csv(args.csv)
    todo = sum(1 for r in rows if not clean(r["applies_to"]))
    table = defaultdict(Counter)
    phases = defaultdict(Counter)
    for r in rows:
        who = clean(r["applies_to"]) or "(not annotated)"
        table[r["keyword"]][who] += 1
        phases[who][r["phase"]] += 1
    print("keyword ; applies_to ; n")
    for kw in table:
        for who, n in table[kw].most_common():
            print(f"{kw} ; {who} ; {n}")
    print("\napplies_to ; " + " ; ".join(list(PHASES.values()) + [OTHER_PHASE]))
    for who, c in phases.items():
        print(f"{who} ; " + " ; ".join(str(c[p]) for p in list(PHASES.values()) + [OTHER_PHASE]))
    shared = [kw for kw in table
              if "Lison" in table[kw] and any(w in table[kw] for w in
                                              ("Séverine", "Flore", "Philomène", "Phasie", "Louisette"))]
    print(f"\nKeywords applied to both Lison and at least one woman: {', '.join(shared) or 'none'}")
    if todo:
        print(f"\nWarning: {todo} rows without applies_to (ignored by the gephi command).")


def gephi(args):
    excluded = {x.strip() for x in args.exclude.split(",") if x.strip()}
    rows = [r for r in read_csv(args.csv)
            if clean(r["applies_to"]) and clean(r["applies_to"]) not in excluded]
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    phase_cols = list(PHASES.values()) + [OTHER_PHASE]

    word_edges = defaultdict(Counter)
    agent_edges = Counter()
    word_chars, agent_chars, words = set(), set(), set()
    for r in rows:
        kw, who = r["keyword"], clean(r["applies_to"])
        words.add(kw)
        word_chars.add(who)
        word_edges[(kw, who)][r["phase"]] += 1
        agent = clean(r["agent"])
        if agent and agent not in excluded:
            agent_chars.update([agent, who])
            agent_edges[(agent, who, kw)] += 1

    # one node file per graph, so that no node is left isolated
    with open(out / "nodes_words.csv", "w", encoding="utf-8", newline="") as f:
        w = csv.writer(f)
        w.writerow(["Id", "Label", "type", "group"])
        for kw in sorted(words):
            w.writerow([f"w_{kw}", kw, "word", WORD_GROUP.get(kw, "")])
        for c in sorted(word_chars):
            w.writerow([f"c_{c}", c, "character", "character"])
    with open(out / "nodes_agents.csv", "w", encoding="utf-8", newline="") as f:
        w = csv.writer(f)
        w.writerow(["Id", "Label", "type"])
        for c in sorted(agent_chars):
            w.writerow([f"c_{c}", c, "character"])

    with open(out / "edges_words.csv", "w", encoding="utf-8", newline="") as f:
        w = csv.writer(f)
        w.writerow(["Source", "Target", "Type", "Weight"] + phase_cols)
        for (kw, who), c in sorted(word_edges.items()):
            w.writerow([f"w_{kw}", f"c_{who}", "Undirected", sum(c.values())] + [c[p] for p in phase_cols])

    with open(out / "edges_agents.csv", "w", encoding="utf-8", newline="") as f:
        w = csv.writer(f)
        w.writerow(["Source", "Target", "Type", "Weight", "Label"])
        for (agent, who, kw), n in sorted(agent_edges.items()):
            w.writerow([f"c_{agent}", f"c_{who}", "Directed", n, kw])

    print(f"Excluded categories: {', '.join(sorted(excluded)) or 'none'}")
    print(f"{len(rows)} rows -> {len(words)} words, {len(word_chars)} characters in the word graph")
    print(f"{len(word_edges)} word edges, {len(agent_edges)} agent edges")
    print(f"Files written to {out}/: nodes_words.csv + edges_words.csv, nodes_agents.csv + edges_agents.csv")


def main(argv=None):
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = p.add_subparsers(dest="command", required=True)
    e = sub.add_parser("extract")
    e.add_argument("--raw", required=True)
    e.add_argument("--csv", required=True)
    s = sub.add_parser("summary")
    s.add_argument("--csv", required=True)
    g = sub.add_parser("gephi")
    g.add_argument("--csv", required=True)
    g.add_argument("--out", required=True)
    g.add_argument("--exclude", default="other",
                   help='applies_to categories left out, comma-separated (default: "other")')
    args = p.parse_args(argv)
    {"extract": extract, "summary": summary, "gephi": gephi}[args.command](args)


if __name__ == "__main__" and "ipykernel" not in sys.modules:
    main()

In [ ]:
"""Creating a lexicon with NRC french"""

import glob
import csv
from collections import Counter
import pandas as pd

# --- Paramètres ---
FICHIER_NRC = "French-NRC-EmoLex.txt"
MOTIF_CORPUS = "Lemmes_Norm_Chapter*-la-bete-humaine-zola.txt"
SEUIL_FREQ = 3                       # fréquence minimale dans le corpus
POS_GARDEES = ["NOUN", "VERB", "ADJ", "ADV"]
STOPLIST = set()                     # à compléter avec tes rejets (ex. {"donner", "croire"})

EMOTIONS = ["anger", "anticipation", "disgust", "fear", "joy",
            "negative", "positive", "sadness", "surprise", "trust"]

# --- 1. Chargement du NRC et dédoublonnage par lemme français ---
nrc = pd.read_csv(FICHIER_NRC, sep="\t", encoding="utf-8")
nrc = nrc.rename(columns={"French Word": "lemme"})
nrc["lemme"] = nrc["lemme"].astype(str).str.strip().str.lower()
nrc = nrc.groupby("lemme")[EMOTIONS].max().reset_index()
print(f"Lemmes NRC uniques : {len(nrc)}")

# --- 2. Chargement du corpus lemmatisé (TagAnt : forme / POS / lemme) ---
frames = []
for chemin in sorted(glob.glob(MOTIF_CORPUS)):
    df = pd.read_csv(chemin, sep="\t", names=["forme", "pos", "lemme"],
                     quoting=csv.QUOTE_NONE, dtype=str,
                     keep_default_na=False, encoding="utf-8")
    df = df[df["forme"].str.strip() != ""]
    frames.append(df)

corpus = pd.concat(frames, ignore_index=True)
corpus["lemme"] = corpus["lemme"].str.lower()

# --- 3. Fréquence et POS dominante de chaque lemme (mots pleins seulement) ---
contenu = corpus[corpus["pos"].isin(POS_GARDEES)]
freq = Counter(contenu["lemme"])
pos_dominante = contenu.groupby("lemme")["pos"].agg(lambda s: s.mode().iat[0])

# --- 4. Sélection des candidats par catégorie ---
def candidats(masque, nom_categorie):
    sub = nrc[masque & nrc["lemme"].isin(freq)].copy()
    sub["freq_corpus"] = sub["lemme"].map(freq)
    sub["pos"] = sub["lemme"].map(pos_dominante)
    sub = sub[(sub["freq_corpus"] >= SEUIL_FREQ) & (~sub["lemme"].isin(STOPLIST))]
    sub["categorie"] = nom_categorie
    sub["decision"] = ""     # à remplir : garder / rejeter / a_verifier
    sub["remarque"] = ""
    return sub

affection = candidats(
    ((nrc["joy"] == 1) | (nrc["trust"] == 1)) & (nrc["positive"] == 1),
    "AFFECT")

violence = candidats(
    ((nrc["anger"] == 1) | (nrc["fear"] == 1) | (nrc["disgust"] == 1))
    & (nrc["negative"] == 1),
    "VIOL")

# --- 5. Export pour validation manuelle ---
colonnes = ["lemme", "categorie", "freq_corpus", "pos", "decision", "remarque"] + EMOTIONS
resultat = (pd.concat([affection, violence])
            .sort_values(["categorie", "freq_corpus"], ascending=[True, False]))
resultat[colonnes].to_csv("candidats_nrc.csv", index=False, encoding="utf-8")

print(resultat.groupby("categorie").size())

"""Count of lexicons' weight"""

"""
Lexicon analysis of La Bête humaine (TagAnt output, French model).

Input : 12 files  Lemmes_Norm_Chapter<N>-....txt  (3 tab-separated columns, no header:
        form / POS tag / lemma)
        + lexiques_documentation.csv (columns: lemme, categorie, niveau, ...)
Output: resultats_par_chapitre.csv, resultats_voisinage_cibles.csv
"""

Lemmes NRC uniques : 11387


ValueError: No objects to concatenate

Count of lexicons' weight

In [41]:
# =====================================================================
# Lexicon analysis of La Bête humaine (paste in ONE Colab cell and run)
#
# Inputs
#   - LEMMA_FOLDER : the TagAnt output files of the chapters
#                    (e.g. Lemmes_Norm_Chapter4-la-bete-humaine-zola.txt)
#   - LEXICON_CSV  : lexiques_documentation.csv
#                    (separator ";", columns lemma, category, level, ...)
#
# Outputs (CSV files written next to the notebook, then shown below)
#   - table2_chapters.csv      lexicon rate per 10,000 tokens, per chapter
#   - table1_neighbourhood.csv ratios around each character (±5 tokens)
#   - top_lemmas.csv           most frequent lexicon lemmas around each character
#   - robustness_core.csv      same ratios with the "core" level only
# =====================================================================

import csv
import glob
import os
import re
from collections import Counter, defaultdict

import pandas as pd

# ----------------------------- CONFIGURATION -----------------------------
LEMMA_FOLDER = "Lemmes_Norm"                       # folder with the TagAnt files
LEMMA_PATTERN = "Lemmes_Norm_Chapter*"   # file name pattern of these files
LEXICON_CSV = "lexiques_documentation.csv"
WINDOW = 5                               # tokens on each side of a character

# Characters, as written in the NORMALIZED text (word form, case-sensitive)
TARGETS = ["Lison", "Séverine", "Flore", "Phasie", "Philomène", "Louisette", "Jacques"]
# -------------------------------------------------------------------------


def read_tagant(path):
    """Return a list of (word, pos, lemma) from a TagAnt output file.
    Handles the two usual TagAnt layouts:
      - one token per line, tab-separated: word <TAB> POS <TAB> lemma
      - running text with tokens written word_POS_lemma
    """
    with open(path, encoding="utf-8", errors="replace") as f:
        text = f.read()
    tokens = []
    lines = [l for l in text.splitlines() if l.strip()]
    if lines and sum(l.count("\t") >= 2 for l in lines[:50]) > 25:
        for l in lines:
            parts = l.split("\t")
            if len(parts) >= 3:
                tokens.append((parts[0], parts[1], parts[2]))
    else:
        for t in text.split():
            parts = t.rsplit("_", 2)
            if len(parts) == 3:
                tokens.append((parts[0], parts[1], parts[2]))
    return tokens


def is_word(word, pos):
    return pos != "PUNCT" and any(c.isalpha() for c in word)


# 1. chapters ---------------------------------------------------------------
files = glob.glob(os.path.join(LEMMA_FOLDER, LEMMA_PATTERN))
if not files:
    raise SystemExit(f"No file matches {LEMMA_PATTERN} in {os.path.abspath(LEMMA_FOLDER)}")

chapters = {}
for path in files:
    m = re.search(r"Chapter\s*(\d+)", os.path.basename(path), re.IGNORECASE)
    if not m:
        print("skipped (no chapter number in the name):", path)
        continue
    toks = [(w, p, l.lower()) for w, p, l in read_tagant(path) if is_word(w, p)]
    if not toks:
        raise SystemExit(f"No token read in {path}: check the TagAnt output format")
    chapters[int(m.group(1))] = toks

total_tokens = sum(len(t) for t in chapters.values())
print(f"{len(chapters)} chapters, {total_tokens} word tokens")
print("example tokens:", chapters[min(chapters)][:5])

# 2. lexicons -----------------------------------------------------------------
doc = pd.read_csv(LEXICON_CSV, sep=";")
doc["lemma"] = doc["lemma"].str.strip().str.lower()
CATEGORIES = list(dict.fromkeys(doc["category"]))           # order of the file
lex_all = {k: set(doc.loc[doc.category == k, "lemma"]) for k in CATEGORIES}
lex_core = {k: set(doc.loc[(doc.category == k) & (doc.level == "core"), "lemma"]) for k in CATEGORIES}
print("lemmas per category:", {k: len(v) for k, v in lex_all.items()})
shared = doc.groupby("lemma")["category"].nunique()
print("lemmas in two categories:", sorted(shared[shared > 1].index))


# 3. Table 2: rate per 10,000 tokens per chapter --------------------------------
rows = []
totals = Counter()
for ch in sorted(chapters):
    toks = chapters[ch]
    row = {"chapter": ch, "tokens": len(toks)}
    for k in CATEGORIES:
        n = sum(1 for _, _, l in toks if l in lex_all[k])
        totals[k] += n
        row[k] = round(n / len(toks) * 10000, 1)
    rows.append(row)
rows.append({"chapter": "total", "tokens": total_tokens,
             **{k: round(totals[k] / total_tokens * 10000, 1) for k in CATEGORIES}})
table2 = pd.DataFrame(rows)
table2.to_csv("table2_chapters.csv", index=False)
print("\nTABLE 2 - lexicon rate per 10,000 tokens")
print(table2.to_string(index=False))


# 4. Table 1: neighbourhood ratios ------------------------------------------------
def neighbourhood(lexicons):
    baseline = {k: sum(1 for t in chapters.values() for _, _, l in t if l in lexicons[k]) / total_tokens
                for k in CATEGORIES}
    out, tops = [], {}
    for target in TARGETS:
        occ = 0
        window_lemmas = []
        for toks in chapters.values():
            positions = [i for i, (w, _, _) in enumerate(toks) if w == target]
            occ += len(positions)
            covered = set()   # union of windows: a token near two mentions counts once
            for i in positions:
                covered.update(range(max(0, i - WINDOW), min(len(toks), i + WINDOW + 1)))
            covered -= set(positions)  # the character's own mentions are not context
            window_lemmas += [toks[j][2] for j in sorted(covered)]
        row = {"character": target, "occurrences": occ, "window_tokens": len(window_lemmas)}
        for k in CATEGORIES:
            hits = [l for l in window_lemmas if l in lexicons[k]]
            row[k + "_n"] = len(hits)
            row[k + "_ratio"] = (round(len(hits) / len(window_lemmas) / baseline[k], 2)
                                 if window_lemmas and baseline[k] else None)
            tops[(target, k)] = Counter(hits).most_common(6)
        out.append(row)
    return pd.DataFrame(out), tops


table1, tops = neighbourhood(lex_all)
table1.to_csv("table1_neighbourhood.csv", index=False)
print(f"\nTABLE 1 - ratios around each character (±{WINDOW} tokens), 1 = chance; counts in brackets")
display_t1 = table1[["character", "occurrences"]].copy()
for k in CATEGORIES:
    display_t1[k] = [f"{r} ({n})" for r, n in zip(table1[k + "_ratio"], table1[k + "_n"])]
print(display_t1.to_string(index=False))
small = table1.loc[table1.occurrences < 30, "character"].tolist()
if small:
    print("Fewer than 30 occurrences, interpret with care:", ", ".join(small))

with open("top_lemmas.csv", "w", encoding="utf-8", newline="") as f:
    w = csv.writer(f)
    w.writerow(["character", "category", "top_lemmas"])
    for (target, k), top in tops.items():
        w.writerow([target, k, ", ".join(f"{l} ({n})" for l, n in top)])
print("\nTop lexicon lemmas around Lison:")
for k in CATEGORIES:
    print(f"  {k}: {', '.join(f'{l} ({n})' for l, n in tops[('Lison', k)]) or '-'}")


# 5. Robustness: core lemmas only ----------------------------------------------------
core, _ = neighbourhood(lex_core)
core.to_csv("robustness_core.csv", index=False)
print("\nROBUSTNESS - same ratios with the core lexicons only")
print(core[["character"] + [k + "_ratio" for k in CATEGORIES]].to_string(index=False))
print("\nFiles written: table2_chapters.csv, table1_neighbourhood.csv, top_lemmas.csv, robustness_core.csv")

12 chapters, 135702 word tokens
example tokens: [('I', 'NOUN', 'i'), ('En', 'ADP', 'en'), ('entrant', 'VERB', 'entrer'), ('dans', 'ADP', 'dans'), ('la', 'DET', 'le')]
lemmas per category: {'affection': 44, 'body': 81, 'desire': 37, 'machine': 55, 'possession': 41, 'violence': 65}
lemmas in two categories: ['blesser', 'blessure', 'cadavre', 'frisson', 'sang', 'sanglant']

TABLE 2 - lexicon rate per 10,000 tokens
chapter  tokens  affection  body  desire  machine  possession  violence
      1   11432       42.9 195.9    27.1    108.5        33.2      72.6
      2   11706       37.6 208.4    33.3    121.3        25.6      86.3
      3    8754       19.4 162.2     4.6     75.4         9.1      48.0
      4   11376       45.7 123.9    22.9     19.3        44.0      95.8
      5   11395       64.9 125.5    27.2     80.7        40.4      39.5
      6   11119       68.4 152.9    43.2     49.5        54.9      83.6
      7   11139       30.5 167.0    13.5    172.4        11.7      45.8
      8  

SpaCy installation

In [2]:
import sys
print(sys.version)

3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]


In [ ]:
!pip install -q spacy
!python -m spacy download fr_core_news_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 MB 47.9 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


Verbs extraction

In [43]:
"""
Verbs of the agents (Lison, Jacques, Séverine, Flore, Phasie) with spaCy, for Gephi.

Input : Norm_Chapter<N>-....txt   (normalised plain text, NOT the TagAnt files)
        lexiques_documentation.csv (columns lemme, categorie, ...)
Output: verbes_agents.csv         one row per (agent, role, verb) occurrence, with the sentence
        verbes_par_agent.csv      counts per agent / role / verb lemma / lexicon category
        gephi_nodes.csv, gephi_edges.csv           agent -> verb -> agent network
        gephi_edges_agents.csv    direct agent -> agent links (subject -> object of the same verb)
Only NAMED agents are used: pronouns (il, elle) are not resolved.
"""
import glob, re
from collections import Counter, defaultdict
import pandas as pd

# ------------------------------------------------------------------ settings
TEXT_DIR = "Norm/"                          # folder with the 12 Norm_Chapter files
LEXICON_CSV = "lexiques_documentation.csv"
OUT_DIR = "./"
AGENTS = {"Lison", "Jacques", "Séverine", "Flore", "Phasie", "Philomène", "Louisette"}
MIN_WEIGHT = 2                              # Gephi: keep verbs used at least twice with an agent
SUBJ_DEPS = {"nsubj"}                       # active subject
PASS_SUBJ_DEPS = {"nsubj:pass"}             # passive subject = the agent is the patient
AGENT_COMPLEMENT = {"obl:agent"}            # "par Jacques" = the agent is the doer
OBJ_DEPS = {"obj", "iobj", "obl:arg"}       # direct / indirect object ("répondre à Lison")

# ------------------------------------------------------------------ 1. load texts
def to_paragraphs(text):
    """One string per paragraph: the texts are hard-wrapped, so join the lines of a paragraph."""
    text = text.replace("--", "— ")                           # dialogue dashes
    paragraphs = re.split(r"\n\s*\n", text)                   # blank line = new paragraph
    return [re.sub(r"\s*\n\s*", " ", p).strip() for p in paragraphs if p.strip()]

chapters = {}
for path in glob.glob(TEXT_DIR + "*Norm_Chapter*.txt"):
    chap = int(re.search(r"Chapter(\d+)", path).group(1))
    with open(path, encoding="utf-8") as f:
        chapters[chap] = to_paragraphs(f.read())
print(len(chapters), "chapters loaded")                       # must be 12

# ------------------------------------------------------------------ 2. parse with spaCy
import spacy
nlp = spacy.load("fr_core_news_md", disable=["ner"])          # parser + tagger + lemmatiser

def agents_of(tokens):
    """Named agents among these tokens, also following 'Jacques et Séverine' (conjuncts)."""
    found = []
    for t in tokens:
        for x in [t] + list(t.conjuncts):
            if x.text in AGENTS:
                found.append(x.text)
    return found

events = []                                                    # one row per (agent, role, verb)
for chap in sorted(chapters):
    for doc in nlp.pipe(chapters[chap], batch_size=50):
        for v in doc:
            if v.pos_ != "VERB":
                continue
            kids = list(v.children)
            subj = [c for c in kids if c.dep_ in SUBJ_DEPS]
            if not subj and v.dep_ == "conj" and v.head.pos_ in {"VERB", "AUX"}:
                subj = [c for c in v.head.children if c.dep_ in SUBJ_DEPS]   # "Il entra et regarda"
            doers = agents_of(subj) + agents_of([c for c in kids if c.dep_ in AGENT_COMPLEMENT])
            patients = agents_of([c for c in kids if c.dep_ in OBJ_DEPS | PASS_SUBJ_DEPS])
            if not doers and not patients:
                continue
            lemma = v.lemma_.lower()
            for a in doers:
                events.append({"chapitre": chap, "agent": a, "role": "sujet", "verbe": lemma,
                               "partenaire": ",".join(patients), "phrase": v.sent.text.strip()})
            for a in patients:
                events.append({"chapitre": chap, "agent": a, "role": "objet", "verbe": lemma,
                               "partenaire": ",".join(doers), "phrase": v.sent.text.strip()})

ev = pd.DataFrame(events)
ev.to_csv(OUT_DIR + "verbes_agents.csv", index=False)
print(len(ev), "events")
print(ev.groupby(["agent", "role"]).size().unstack(fill_value=0))

# ------------------------------------------------------------------ 3. lexicon category of each verb
doc_lex = pd.read_csv(LEXICON_CSV, sep=";")
cat_of = dict(zip(doc_lex.lemma.str.strip().str.lower(), doc_lex.category))
ev["categorie"] = ev.verbe.map(cat_of).fillna("autre")

summary = (ev.groupby(["agent", "role", "verbe", "categorie"]).size()
             .reset_index(name="n").sort_values(["agent", "role", "n"], ascending=[True, True, False]))
summary.to_csv(OUT_DIR + "verbes_par_agent.csv", index=False)
for (agent, role), g in summary.groupby(["agent", "role"]):
    print(agent, role, g.head(8)[["verbe", "n"]].values.tolist())

# ------------------------------------------------------------------ 4. Gephi files
# agent --(subject)--> verb --(object)--> agent : "Jacques -> regarder -> Lison"
edges = ev.groupby(["agent", "role", "verbe"]).size().reset_index(name="Weight")
edges = edges[edges.Weight >= MIN_WEIGHT]
rows = []
for r in edges.itertuples():
    verb_id = "v:" + r.verbe
    if r.role == "sujet":
        rows.append({"Source": r.agent, "Target": verb_id, "Weight": r.Weight, "Type": "Directed", "Role": "sujet"})
    else:
        rows.append({"Source": verb_id, "Target": r.agent, "Weight": r.Weight, "Type": "Directed", "Role": "objet"})
pd.DataFrame(rows).to_csv(OUT_DIR + "gephi_edges.csv", index=False)

nodes = [{"Id": a, "Label": a, "Type": "agent", "Categorie": "agent"} for a in sorted(AGENTS)]
for verb in sorted(edges.verbe.unique()):
    nodes.append({"Id": "v:" + verb, "Label": verb, "Type": "verbe", "Categorie": cat_of.get(verb, "autre")})
pd.DataFrame(nodes).to_csv(OUT_DIR + "gephi_nodes.csv", index=False)

# direct agent -> agent links (e.g. Jacques -> Séverine), weight = number of shared verbs
direct = Counter()
verbs_of = defaultdict(list)
for e in events:
    if e["role"] == "sujet" and e["partenaire"]:
        for p in e["partenaire"].split(","):
            if p != e["agent"]:
                direct[(e["agent"], p)] += 1
                verbs_of[(e["agent"], p)].append(e["verbe"])
pd.DataFrame([{"Source": s, "Target": t, "Weight": w, "Type": "Directed",
               "Label": " ".join(sorted(set(verbs_of[(s, t)])))} for (s, t), w in direct.items()]
             ).to_csv(OUT_DIR + "gephi_edges_agents.csv", index=False)
print("Gephi files written")

12 chapters loaded
565 events
role       objet  sujet
agent                  
Flore         15     39
Jacques       62    199
Lison         30     48
Louisette      4      4
Phasie         3      3
Philomène      6     19
Séverine      66     67
Flore objet [['heurter', 2], ['reconnaître', 2], ['charger', 1], ['crier', 1], ['dire', 1], ['débarrasser', 1], ['fixer', 1], ['planter', 1]]
Flore sujet [['dire', 3], ['tourner', 3], ['regarder', 2], ['reprendre', 2], ['apercevoir', 1], ['avoir', 1], ['blêmir', 1], ['bouger', 1]]
Jacques objet [['tourner', 4], ['apercevoir', 3], ['dire', 3], ['donner', 3], ['surprendre', 3], ['demander', 2], ['parler', 2], ['répondre', 2]]
Jacques sujet [['dire', 12], ['devoir', 8], ['regarder', 8], ['avoir', 7], ['pouvoir', 7], ['voir', 6], ['crier', 5], ['trouver', 5]]
Lison objet [['pousser', 3], ['approcher', 2], ['arriver', 2], ['dégager', 2], ['quitter', 2], ['toucher', 2], ['accommoder', 1], ['aimer', 1]]
Lison sujet [['arrêter', 3], ['continuer', 3], [

Creation of a research expression from lexicons



In [ ]:
for cat in ["affection", "desir", "corps", "possession", "violence", "machine"]:
    mots = [l.strip().lower() for l in open(f"lexiques/{cat}.txt", encoding="utf-8") if l.strip()]
    print(cat, "→", r"\b(" + "|".join(sorted(mots)) + r")\b")

affection → \b(adorable|adoration|adorer|affection|aimable|aimer|ami|amie|amitié|amour|amoureux|attentif|bonheur|camarade|caressant|caresse|chéri|compagnon|confiance|consolation|consoler|câlin|dou|douceur|doux|dévouement|entente|familier|fidèle|félicité|gentil|gentille|joie|mignon|pitié|plaisir|protecteur|protéger|rassurer|reconnaissance|soigner|soin|sourire|tendresse)\b
desir → \b(amant|ardent|ardente|baisa|baiser|convoitise|débauche|désir|désirer|désireux|embrasser|envie|excitation|exciter|femelle|frisson|frissonner|frémir|frémissement|ivresse|jouir|jouissance|mâle|nudité|passion|rut|sensuel|sexe|séduction|séduire|vice|vierge|virginité|voluptueux|volupté|étreindre|étreinte)\b
corps → \b(barbe|blême|bouche|bras|cerveau|chair|chaleur|chevelure|cheveu|coeur|corps|cou|coude|crâne|cuisse|dent|doigt|dos|entrailles|flanc|front|genou|gorge|haleine|hanche|jambe|joue|livide|lèvre|membre|muscle|nerf|nuque|os|peau|pied|plaie|poing|poitrine|pâleur|rein|sein|souffle|spasme|sueur|tiédeur|toux|veine

Test with Pronouns

In [ ]:
"""
Verbs of the agents (Lison, Jacques, Séverine, Flore, Phasie) with spaCy, for Gephi.
Version 2: the subjects "il" / "elle" are resolved to a named character with a simple rule.

Input : Norm_Chapter<N>-....txt   (normalised plain text, NOT the TagAnt files)
        lexiques_documentation.csv (columns lemme, categorie, ...)
Output: verbes_agents.csv         one row per (agent, role, verb) occurrence, with the sentence
        verbes_par_agent.csv      counts per agent / role / verb lemma / lexicon category
        verification_pronoms.csv  60 random resolved pronouns to check BY HAND
        gephi_nodes.csv, gephi_edges.csv, gephi_edges_agents.csv

Pronoun rule (a heuristic, NOT a real coreference model):
  "il"   -> a male character, "elle" -> a female character (Lison counts as female)
  antecedent = the nearest earlier subject of the right gender in the same sentence,
  else the nearest earlier mention in the same sentence,
  else the subject (then the last mention) of that gender in the previous sentences.
"""
import glob, re
from collections import Counter, defaultdict
import pandas as pd

# ------------------------------------------------------------------ settings
TEXT_DIR = "Norm/"                          # folder with the 12 Norm_Chapter files
LEXICON_CSV = "lexiques_documentation.csv"
OUT_DIR = "./"
AGENTS = {"Lison", "Jacques", "Séverine", "Flore", "Phasie"}
# ALL named characters that can be the antecedent of il/elle (other characters must be here,
# otherwise "il" after "Roubaud" would wrongly be attributed to Jacques). Add names as needed.
MALE = {"Jacques", "Roubaud", "Pecqueux", "Cabuche", "Grandmorin", "Misard", "Moulin",
        "Dabadie", "Cauche", "Denizet", "Camy-Lamotte", "Lachesnaye"}
FEMALE = {"Séverine", "Flore", "Phasie", "Lison", "Philomène", "Victoire", "Guichon"}
PRONOUNS = {"il": "M", "elle": "F"}
IMPERSONAL = {"falloir", "pleuvoir", "neiger"}   # "il faut", "il pleut": no antecedent
MIN_WEIGHT = 2                              # Gephi: keep verbs used at least twice with an agent
SUBJ_DEPS = {"nsubj"}
PASS_SUBJ_DEPS = {"nsubj:pass"}
AGENT_COMPLEMENT = {"obl:agent"}
OBJ_DEPS = {"obj", "iobj", "obl:arg"}

# ------------------------------------------------------------------ 1. load texts
def to_paragraphs(text):
    """One string per paragraph: the texts are hard-wrapped, so join the lines of a paragraph."""
    text = text.replace("--", "— ")                           # dialogue dashes
    paragraphs = re.split(r"\n\s*\n", text)                   # blank line = new paragraph
    return [re.sub(r"\s*\n\s*", " ", p).strip() for p in paragraphs if p.strip()]

chapters = {}
for path in glob.glob(TEXT_DIR + "*Norm_Chapter*.txt"):
    chap = int(re.search(r"Chapter(\d+)", path).group(1))
    with open(path, encoding="utf-8") as f:
        chapters[chap] = to_paragraphs(f.read())
print(len(chapters), "chapters loaded")                       # must be 12

# ------------------------------------------------------------------ 2. parse with spaCy
import spacy
nlp = spacy.load("fr_core_news_md", disable=["ner"])          # parser + tagger + lemmatiser

def agents_of(tokens):
    """Named agents among these tokens, also following 'Jacques et Séverine' (conjuncts)."""
    found = []
    for t in tokens:
        for x in [t] + list(t.conjuncts):
            if x.text in AGENTS:
                found.append(x.text)
    return found

def named_people(sent):
    """Named characters of the sentence in reading order: [(token, 'M' or 'F')]."""
    people = []
    for t in sent:
        if t.text in MALE | FEMALE and not (t.i > 0 and t.doc[t.i - 1].lower_ in {"les", "des"}):
            people.append((t, "M" if t.text in MALE else "F"))   # "les Roubaud" = plural, skipped
    return people

def antecedent(pron, gender, people, memory):
    """Antecedent of an 'il'/'elle' (see the rule in the header)."""
    before = [t for t, g in people if g == gender and t.i < pron.i]
    subjects = [t for t in before if t.dep_ in SUBJ_DEPS | PASS_SUBJ_DEPS]
    if subjects:
        return subjects[-1].text
    if before:
        return before[-1].text
    return memory[gender]["subj"] or memory[gender]["last"]

def update(memory, people):
    """Remember who was the last subject / the last mention of each gender."""
    for t, g in people:
        memory[g]["last"] = t.text
        if t.dep_ in SUBJ_DEPS | PASS_SUBJ_DEPS:
            memory[g]["subj"] = t.text

events = []                                                    # one row per (agent, role, verb)
for chap in sorted(chapters):
    memory = {"M": {"subj": None, "last": None}, "F": {"subj": None, "last": None}}
    prev_sentence = ""
    for doc in nlp.pipe(chapters[chap], batch_size=50):
        for sent in doc.sents:
            people = named_people(sent)
            for v in sent:
                if v.pos_ != "VERB" or v.lemma_.lower() in IMPERSONAL:
                    continue
                kids = list(v.children)
                subj = [c for c in kids if c.dep_ in SUBJ_DEPS]
                if not subj and v.dep_ == "conj" and v.head.pos_ in {"VERB", "AUX"}:
                    subj = [c for c in v.head.children if c.dep_ in SUBJ_DEPS]   # "Il entra et regarda"
                doers = []                                     # [(agent, how it was found)]
                for s in subj:
                    for x in [s] + list(s.conjuncts):
                        if x.text in AGENTS:
                            doers.append((x.text, "nom"))
                        elif x.lower_ in PRONOUNS and x.pos_ == "PRON":
                            who = antecedent(x, PRONOUNS[x.lower_], people, memory)
                            if who in AGENTS:
                                doers.append((who, "pronom:" + x.lower_))
                doers += [(n, "nom") for n in agents_of([c for c in kids if c.dep_ in AGENT_COMPLEMENT])]
                patients = agents_of([c for c in kids if c.dep_ in OBJ_DEPS | PASS_SUBJ_DEPS])
                if not doers and not patients:
                    continue
                lemma = v.lemma_.lower()
                for a, how in doers:
                    events.append({"chapitre": chap, "agent": a, "role": "sujet", "verbe": lemma,
                                   "methode": how, "partenaire": ",".join(patients),
                                   "phrase_precedente": prev_sentence, "phrase": sent.text.strip()})
                for a in patients:
                    events.append({"chapitre": chap, "agent": a, "role": "objet", "verbe": lemma,
                                   "methode": "nom", "partenaire": ",".join(d for d, _ in doers),
                                   "phrase_precedente": prev_sentence, "phrase": sent.text.strip()})
            update(memory, people)                             # after the sentence, not before
            prev_sentence = sent.text.strip()

ev = pd.DataFrame(events)
ev.to_csv(OUT_DIR + "verbes_agents.csv", index=False)
print(len(ev), "events")
print(ev.groupby(["agent", "role"]).size().unstack(fill_value=0))
print(ev.methode.str.split(":").str[0].value_counts())        # found by name vs by pronoun

# sample of resolved pronouns: fill the last column by hand (oui/non) to measure the accuracy
resolved = ev[ev.methode.str.startswith("pronom")]
sample = resolved.sample(min(60, len(resolved)), random_state=1)[
    ["chapitre", "methode", "agent", "verbe", "phrase_precedente", "phrase"]]
sample["correct (oui/non)"] = ""
sample.to_csv(OUT_DIR + "verification_pronoms.csv", index=False)

# ------------------------------------------------------------------ 3. lexicon category of each verb
doc_lex = pd.read_csv(LEXICON_CSV)
cat_of = dict(zip(doc_lex.lemme.str.lower(), doc_lex.categorie))
ev["categorie"] = ev.verbe.map(cat_of).fillna("autre")

summary = (ev.groupby(["agent", "role", "verbe", "categorie"]).size()
             .reset_index(name="n").sort_values(["agent", "role", "n"], ascending=[True, True, False]))
summary.to_csv(OUT_DIR + "verbes_par_agent.csv", index=False)
for (agent, role), g in summary.groupby(["agent", "role"]):
    print(agent, role, g.head(8)[["verbe", "n"]].values.tolist())

# ------------------------------------------------------------------ 4. Gephi files
edges = ev.groupby(["agent", "role", "verbe"]).size().reset_index(name="Weight")
edges = edges[edges.Weight >= MIN_WEIGHT]
rows = []
for r in edges.itertuples():
    verb_id = "v:" + r.verbe
    if r.role == "sujet":
        rows.append({"Source": r.agent, "Target": verb_id, "Weight": r.Weight, "Type": "Directed", "Role": "sujet"})
    else:
        rows.append({"Source": verb_id, "Target": r.agent, "Weight": r.Weight, "Type": "Directed", "Role": "objet"})
pd.DataFrame(rows).to_csv(OUT_DIR + "gephi_edges.csv", index=False)

nodes = [{"Id": a, "Label": a, "Type": "agent", "Categorie": "agent"} for a in sorted(AGENTS)]
for verb in sorted(edges.verbe.unique()):
    nodes.append({"Id": "v:" + verb, "Label": verb, "Type": "verbe", "Categorie": cat_of.get(verb, "autre")})
pd.DataFrame(nodes).to_csv(OUT_DIR + "gephi_nodes.csv", index=False)

direct = Counter()
verbs_of = defaultdict(list)
for e in events:
    if e["role"] == "sujet" and e["partenaire"]:
        for p in e["partenaire"].split(","):
            if p != e["agent"]:
                direct[(e["agent"], p)] += 1
                verbs_of[(e["agent"], p)].append(e["verbe"])
pd.DataFrame([{"Source": s, "Target": t, "Weight": w, "Type": "Directed",
               "Label": " ".join(sorted(set(verbs_of[(s, t)])))} for (s, t), w in direct.items()]
             ).to_csv(OUT_DIR + "gephi_edges_agents.csv", index=False)
print("Gephi files written")

12 chapters loaded
3204 events
role      objet  sujet
agent                 
Flore        14    320
Jacques      62   1335
Lison        34    279
Phasie        0     81
Séverine     66   1013
methode
pronom    2665
nom        539
Name: count, dtype: int64
Flore objet [['heurter', 2], ['reconnaître', 2], ['charger', 1], ['crier', 1], ['dire', 1], ['débarrasser', 1], ['fixer', 1], ['planter', 1]]
Flore sujet [['avoir', 22], ['pouvoir', 9], ['vouloir', 8], ['faire', 7], ['venir', 7], ['voir', 7], ['dire', 6], ['apercevoir', 5]]
Jacques objet [['tourner', 4], ['apercevoir', 3], ['dire', 3], ['donner', 3], ['surprendre', 3], ['demander', 2], ['parler', 2], ['répondre', 2]]
Jacques sujet [['avoir', 85], ['faire', 39], ['voir', 38], ['vouloir', 35], ['savoir', 31], ['sentir', 31], ['devoir', 27], ['pouvoir', 27]]
Lison objet [['pousser', 3], ['apercevoir', 2], ['approcher', 2], ['arriver', 2], ['dégager', 2], ['quitter', 2], ['sauter', 2], ['toucher', 2]]
Lison sujet [['avoir', 13], ['rouler'

Generating files with lemmas in text without punctuation or space

In [ ]:
import glob, os, csv
import pandas as pd

os.makedirs("Lemmes_txt", exist_ok=True)
for p in glob.glob("Lemmes_Norm/Lemmes_Norm_Chapter*.txt"):
    d = pd.read_csv(p, sep="\t", names=["forme", "pos", "lemme"],
                    quoting=csv.QUOTE_NONE, dtype=str, keep_default_na=False)
    d = d[~d.pos.isin(["PUNCT", "SPACE"]) & (d.forme.str.strip() != "")]
    out = "Lemmes_txt/" + os.path.basename(p).replace("Lemmes_Norm", "Lemmes_txt")
    open(out, "w", encoding="utf-8").write(" ".join(d.lemme.str.lower()))

Downloading FEEL lexicon with R

In [ ]:
install.packages("remotes")
remotes::install_github("colinfay/rfeel")

library(rfeel)
feel <- rfeel::rfeel("score"
head(feel)
write.csv(feel, "feel_score.csv", row.names = FALSE)

SyntaxError: invalid syntax (2401287309.py, line 2)

Comapring with my lexicons

In [ ]:
import pandas as pd
feel = pd.read_csv("feel_score.csv")
doc  = pd.read_csv("lexiques_documentation.csv")

# adapte "word" au nom réel de la colonne des mots dans feel
feel["word"] = feel["word"].str.lower()
doc["dans_FEEL"] = doc["lemme"].str.lower().isin(set(feel["word"]))
print(doc.groupby("categorie")["dans_FEEL"].agg(["sum","count"]))

            sum  count
categorie             
affection    15     44
corps        13     51
desir        14     37
machine      18     55
possession   20     41
violence     51     65
